In [1]:
import sys
sys.path.append("LIBS")

import torch
from LIBS.PIGVAE import *
from LIBS.Putils import *
from LIBS.P_egnn_clean import EGNN
from LIBS.create_full_graph_data_physics import *
from LIBS.force_field import *

/home/clod/anaconda3/envs/openmm-dev/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


![GVAE Architecture Diagram](https://storage.googleapis.com/aai-web-images/colab/molecule-gvae-architecture.png)

In [5]:
dataset = get_dataset()

Loading dataset ...
Root directory: None
TPR file: MD.tpr
Trajectory file: MD_with_solvent_noPBC.xtc
Selection: protein
Include atom type: True
Scale features: True
Scale positions: True
Initial alignment: False
Return max position: False
Return position in Angstrom: True
No root directory provided, using default path ...
Successfully loaded processed data from /home/clod/Desktop/THESIS_2/PIGVAE/DATA/processed/data_MD_protein.pt
Positions are already in Angstroms, no conversion needed ...

Scaling positions... Max absolute position values for scaling: tensor([28.9800, 29.0000, 29.0100])
Positions shape: torch.Size([20032, 52, 3])
Including atom features: performing one hot encoding ...
Scaling features ...
not aligning the frames ...
Dataset created with 20032 graphs.
Number of graphs in the dataset: 20032
Number of features in the dataset: 6
Number of edges in the dataset: 102
Number of nodes in the dataset: 52
Number of features in the dataset: torch.Size([52, 6])



In [7]:
pdbfile = "../DATA/raw/protein_only.pdb"
physics_critic = EnergyCalculator(pdbfile)

Using OpenMM platform: CUDA


In [8]:
def add_physics_attributes_to_dataset(dataset, physics_critic):
    """
    Adds bond parameters (r0, k) as edge attributes to each graph in the dataset.

    This function iterates through the dataset, and for each graph, it creates
    an `edge_attr` tensor where each row corresponds to an edge in `edge_index`
    and contains the physical parameters for that bond.
    """
    print("Adding physics parameters as edge attributes...")

    # Step 1: Create a lookup dictionary for efficient parameter access.
    # The key is a sorted tuple of atom indices for a bond.
    bond_params_lookup = {}
    for i, bond_indices in enumerate(physics_critic.bonds):
        p1, p2 = bond_indices
        key = tuple(sorted((p1, p2))) # Use a sorted tuple to handle undirected edges
        r0 = physics_critic.r0_list[i]
        k = physics_critic.k_list[i]
        bond_params_lookup[key] = [r0, k]

    # Step 2: Iterate through each graph and build its edge_attr tensor.
    new_data_list = []
    for i, data in enumerate(tqdm(dataset, desc="Processing graphs")):
        edge_attributes = []
        for j in range(data.edge_index.size(1)):
            u, v = data.edge_index[:, j].tolist()
            key = tuple(sorted((u, v)))

            # Get the parameters for this specific edge.
            # If an edge exists in the graph but not the topology, use defaults.
            params = bond_params_lookup.get(key, [0.0, 0.0])
            edge_attributes.append(params)

        # Create the final tensor and add it to the data object.
        data.edge_attr = torch.tensor(edge_attributes, dtype=torch.float32)
        new_data_list.append(data)

    # Re-collate the dataset with the new attribute
    new_dataset = InMemoryDataset(root=dataset.root, transform=None)
    new_dataset.data, new_dataset.slices = new_dataset.collate(new_data_list)
    
    print(f"Finished. Edge attributes added with shape: {new_dataset[0].edge_attr.shape}")
    return new_dataset


In [9]:
ph_dataset = add_physics_attributes_to_dataset(dataset, physics_critic)

Adding physics parameters as edge attributes...


Processing graphs: 100%|██████████| 20032/20032 [00:07<00:00, 2730.32it/s]


Finished. Edge attributes added with shape: torch.Size([102, 2])


<bound method BaseData.edge_attrs of Data(x=[52, 6], edge_index=[2, 102], pos=[52, 3], edge_attr=[102, 2])>